# Phần 4 — Kiến trúc kết hợp và Cách Embed ảnh khác nhau

**Khám phá 3 cách tokenize ảnh cho Transformer:**

| Kiến trúc | Token = ? | Số tokens | Feature dim | Seq len |
|-----------|-----------|-----------|-------------|---------|
| **4A: CNN+Transformer** | CNN feature positions | 64 | 64 (CNN) | 64 |
| **4B: Spatial Tokens** | Mỗi pixel (H×W) | 1024 | 3 (RGB) | 1024 |
| **4C: Channel Tokens** | Mỗi channel | 64 | 1024 (spatial) | 64 |

**Câu hỏi:** Cách tokenize nào giúp Transformer học tốt nhất?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

from src.data import get_cifar100_loaders, get_device
from src.models_part4 import CNNTransformerHybrid, SpatialTokenViT, ChannelTokenViT
from src.train import fit, load_best_model
from src.utils import (get_param_count, get_predictions, compute_metrics,
                       plot_multi_curves, plot_comparison_bar, print_results_table,
                       save_metrics_json, load_metrics_json)

DEVICE = get_device()
print(f"Thiết bị: {DEVICE}")
train_loader, val_loader, test_loader, class_names = get_cifar100_loaders(batch_size=128)

## 1. Diagram: So sánh 3 cách tokenize

In [ ]:
# Visualize 3 cách tokenize trên 1 ảnh mẫu
images, labels = next(iter(test_loader))
img = images[0]  # [3, 32, 32]
CIFAR100_MEAN = torch.tensor([0.5071, 0.4867, 0.4408])
CIFAR100_STD  = torch.tensor([0.2675, 0.2565, 0.2761])
img_show = (img * CIFAR100_STD[:, None, None] + CIFAR100_MEAN[:, None, None]).clamp(0,1)
img_np = img_show.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# Original
axes[0].imshow(img_np)
axes[0].set_title(f'Ảnh gốc\n{class_names[labels[0].item()]}')
axes[0].axis('off')

# 4A: CNN patches (8×8 grid = 64 spatial positions)
axes[1].imshow(img_np)
for i in range(8):
    for j in range(8):
        rect = patches.Rectangle((j*4-0.5, i*4-0.5), 4, 4,
                                   linewidth=0.5, edgecolor='red', facecolor='none')
        axes[1].add_patch(rect)
axes[1].set_title('4A: CNN Features\n8×8=64 spatial tokens')
axes[1].axis('off')

# 4B: Spatial tokens (every pixel)
axes[2].imshow(img_np)
for i in range(0, 32, 4):  # Chỉ vẽ mỗi 4 pixel để thấy rõ
    for j in range(0, 32, 4):
        rect = patches.Rectangle((j-0.5, i-0.5), 1, 1,
                                   linewidth=0.5, edgecolor='blue', facecolor='blue', alpha=0.3)
        axes[2].add_patch(rect)
axes[2].set_title('4B: Spatial Tokens\n32×32=1024 pixel tokens')
axes[2].axis('off')

# 4C: Channel tokens
channel_names = ['R', 'G', 'B']
channel_imgs = [img_np[:,:,c] for c in range(3)]
axes[3].axis('off')
axes[3].set_title('4C: Channel Tokens\n64 channel tokens')
inset_positions = [(0.05, 0.05), (0.37, 0.05), (0.69, 0.05)]
for pos, ch_img, ch_name in zip(inset_positions, channel_imgs, channel_names):
    inset = axes[3].inset_axes([pos[0], pos[1], 0.28, 0.9])
    inset.imshow(ch_img, cmap='gray')
    inset.set_title(ch_name, fontsize=8)
    inset.axis('off')

plt.suptitle('3 cách tokenize ảnh cho Transformer', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Kiến trúc 4A: CNN + Transformer Hybrid

In [ ]:
model_4a = CNNTransformerHybrid(num_classes=100)
print(f"CNNTransformerHybrid — Params: {get_param_count(model_4a)}")

# Trace shape
x_dummy = torch.randn(2, 3, 32, 32)
with torch.no_grad():
    cnn_out = model_4a.cnn(x_dummy)
    print(f"\nSau CNN backbone: {cnn_out.shape}  [B, 64, 8, 8]")
    B, C, H, W = cnn_out.shape
    tokens = cnn_out.reshape(B, C, H*W).permute(0, 2, 1)
    print(f"Sau reshape:      {tokens.shape}  [B, 64 tokens, 64 features]")
    tokens_proj = model_4a.token_proj(tokens)
    print(f"Sau projection:   {tokens_proj.shape}  [B, 64 tokens, 128 d_model]")
    out = model_4a(x_dummy)
    print(f"Output:           {out.shape}")

## 3. Kiến trúc 4B: Spatial Token ViT

⚠️ **Cảnh báo bộ nhớ:** 1024 tokens → attention matrix [B, H, 1024, 1024]
Với batch=32, 4 heads: ~512MB RAM!
Sử dụng `batch_size=32` cho mô hình này.

In [ ]:
model_4b = SpatialTokenViT(num_classes=100)
print(f"SpatialTokenViT — Params: {get_param_count(model_4b)}")

# Trace shape
with torch.no_grad():
    B, C, H, W = x_dummy.shape
    tokens = x_dummy.reshape(B, C, H*W).permute(0, 2, 1)
    print(f"\nSau reshape: {tokens.shape}  [B, 1024 pixel tokens, 3 RGB features]")
    tokens_proj = model_4b.pixel_proj(tokens)
    print(f"Sau projection: {tokens_proj.shape}  [B, 1024, 64 d_model]")
    out = model_4b(x_dummy)
    print(f"Output: {out.shape}")

# Tính attention matrix size
attn_size_MB = 32 * 4 * 1024 * 1024 * 4 / 1024**2
print(f"\n⚠️ Attention matrix size (batch=32, 4 heads): {attn_size_MB:.0f}MB")
print("→ Dùng batch_size=32 cho mô hình này")

In [ ]:
# Spatial loader với batch_size=32
from src.data import get_cifar100_loaders
train_loader_small, val_loader_small, test_loader_small, _ = get_cifar100_loaders(batch_size=32)

## 4. Kiến trúc 4C: Channel Token ViT

In [ ]:
model_4c = ChannelTokenViT(num_classes=100)
print(f"ChannelTokenViT — Params: {get_param_count(model_4c)}")

with torch.no_grad():
    x_expanded = model_4c.channel_expand(x_dummy)
    print(f"\nSau Conv 1×1 expand: {x_expanded.shape}  [B, 64 channels, 32, 32]")
    B, C, H, W = x_expanded.shape
    tokens = x_expanded.reshape(B, C, H*W)
    print(f"Sau reshape:         {tokens.shape}  [B, 64 channel tokens, 1024 spatial features]")
    tokens_proj = model_4c.spatial_proj(tokens)
    print(f"Sau projection:      {tokens_proj.shape}  [B, 64, 128 d_model]")
    out = model_4c(x_dummy)
    print(f"Output:              {out.shape}")

## 5. Huấn luyện và So sánh

In [ ]:
TRAIN_MODE = True
histories_p4 = {}

configs_p4 = {
    "CNNTransformerHybrid": {
        "model": CNNTransformerHybrid(100),
        "loader": (train_loader, val_loader),
        "epochs": 50, "lr": 1e-3,
        "ckpt": '../exercise/results/checkpoints/cnn_transformer.pt',
    },
    "SpatialTokenViT": {
        "model": SpatialTokenViT(100),
        "loader": (train_loader_small, val_loader_small),
        "epochs": 50, "lr": 1e-4,
        "ckpt": '../exercise/results/checkpoints/spatial_vit.pt',
    },
    "ChannelTokenViT": {
        "model": ChannelTokenViT(100),
        "loader": (train_loader, val_loader),
        "epochs": 50, "lr": 3e-4,
        "ckpt": '../exercise/results/checkpoints/channel_vit.pt',
    },
}

for name, cfg in configs_p4.items():
    ckpt_path = cfg["ckpt"]
    history_path = f'../exercise/results/metrics/{name.lower()}_history.json'
    tr_loader, vl_loader = cfg["loader"]

    if TRAIN_MODE:
        print(f"\n{'='*50}\nTraining: {name}\n{'='*50}")
        m = cfg["model"].to(DEVICE)
        hist = fit(m, tr_loader, vl_loader,
                   {"epochs": cfg["epochs"], "lr": cfg["lr"],
                    "device": DEVICE, "save_path": ckpt_path})
        histories_p4[name] = hist
        save_metrics_json(hist, history_path)
    else:
        histories_p4[name] = load_metrics_json(history_path)

    print(f"✓ {name} done")

In [ ]:
# So sánh val_acc
plot_multi_curves(
    list(histories_p4.values()),
    list(histories_p4.keys()),
    title="So sánh 3 kiến trúc tokenization (Phần 4)",
    save_path='../exercise/results/plots/part4_comparison_curves.png'
)

In [ ]:
# Bảng kết quả
results_p4 = {}
model_classes = {
    "CNNTransformerHybrid": CNNTransformerHybrid,
    "SpatialTokenViT": SpatialTokenViT,
    "ChannelTokenViT": ChannelTokenViT,
}
test_loaders = {
    "CNNTransformerHybrid": test_loader,
    "SpatialTokenViT": test_loader_small,
    "ChannelTokenViT": test_loader,
}

for name, cls in model_classes.items():
    ckpt = configs_p4[name]["ckpt"]
    if os.path.exists(ckpt):
        m = load_best_model(cls(100), ckpt, DEVICE)
        preds, labels = get_predictions(m, test_loaders[name], DEVICE)
        metrics = compute_metrics(preds, labels)
        results_p4[name] = {
            "test_acc": metrics["accuracy"],
            "val_acc": max(histories_p4[name]["val_acc"]),
            "f1_macro": metrics["f1_macro"],
            "params": get_param_count(m),
        }

save_metrics_json(results_p4, '../exercise/results/metrics/part4_results.json')
print_results_table(results_p4)

plot_comparison_bar(results_p4, metric="test_acc",
                    title="Test Accuracy — Phần 4",
                    save_path='../exercise/results/plots/part4_bar.png')

## 6. Nhận xét

**Phân tích:**

1. **CNNTransformerHybrid** thường đạt kết quả tốt nhất vì:
   - CNN cung cấp đặc trưng phân cấp (hierarchical features)
   - Transformer xử lý quan hệ toàn cục giữa các vùng feature
   - Số tokens vừa phải (64) → attention hiệu quả

2. **SpatialTokenViT** (1024 tokens) gặp vấn đề:
   - Attention matrix O(1024²) = 1M operations mỗi layer → chậm, tốn RAM
   - Với ảnh 32×32, mỗi token (pixel) chỉ có 3 features → feature quá thô
   - Đây là lý do ViT gốc dùng patches thay vì pixels!

3. **ChannelTokenViT** (64 tokens):
   - Tokens biểu diễn "đặc trưng kênh" thay vì "vùng không gian"
   - Có thể học được "channel attention" (channel nào quan trọng hơn)
   - Nhưng mất đi thông tin vị trí không gian

**Kết luận:** Hybrid CNN+Transformer thường cân bằng tốt nhất giữa
đặc trưng cục bộ (CNN) và quan hệ toàn cục (Transformer).